# نوفا الصغير — المرحلة الثانية: إضافة الصورة والصوت (Vision + Audio)

هذا الدفتر **يكمل** تدريب النموذج الذي أنتجه دفتر المرحلة الأولى (النص) — لا يبدأ من الصفر، ولا يفقد أي شيء تعلّمه النموذج من النص. يضيف عليه فقط قدرتين حقيقيتين جديدتين:
- **الصورة**: توليد صور من نص (Generation) + وصف/فهم صورة معطاة (Understanding) — بالاتجاهين معاً من نفس البيانات.
- **الصوت**: تحويل نص لصوت (Generation) + تفريغ/فهم صوت معطى (Understanding) — بالاتجاهين معاً أيضاً.

**نطاق حقيقي يجب معرفته بصراحة:** بيانات الصور المتاحة بكثرة وجاهزة للسحب الآن هي بتعليقات **إنجليزية** (Flickr30k) — لا يوجد مصدر عربي مكافئ بنفس الحجم متاح فوراً. هذا لا يضر الآلية نفسها (تعلّم ربط البكسل برموز، وربطها بنص التعليق) لأنها مستقلة عن اللغة تماماً، لكن يعني أن وصف الصور بالعربية الفصيحة يحتاج بيانات عربية لاحقاً لتحسينه أكثر. الصوت بالمقابل عربي حقيقي بالكامل (Common Voice العربية).

## قبل الضغط على "Save Version → Save & Run All" — 3 خطوات:

1. **أضف نتاج (Output) دفتر المرحلة الأولى كمدخل هنا:** من القائمة الجانبية اضغط **+ Add Input → Notebook Output**، واختر دفتر `sham_small_training` (مرحلة النص) الذي انتهى تدريبه. هذا يجعل نقطة الحفظ وأداة تقسيم النص من المرحلة الأولى متاحتين هنا.
2. **نفس أسرار Kaggle المستخدمة سابقاً:** تأكد أن `GITHUB_TOKEN` لا يزال مضافاً في Add-ons → Secrets (لا حاجة لإضافته من جديد إن كان موجوداً من قبل).
3. **فعّل الإنترنت واختر GPU** من Settings، تماماً كما في المرحلة الأولى.

**مهم:** استخدم **Save Version → Save & Run All (Commit)** مباشرة من البداية لهذا الدفتر (وليس التشغيل التفاعلي) — تعلمنا من المرحلة الأولى أن هذا هو الأسلوب الآمن الوحيد لتشغيل يستغرق ساعات دون قلق من إغلاق الهاتف أو انقطاع الاتصال.


### 1) سحب الكود الحقيقي من GitHub مباشرة (نفس أسلوب المرحلة الأولى)

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

# حقن المسارات ديناميكياً لتفادي ModuleNotFoundError
possible_paths = [
    "/kaggle/working",
    CLONE_DIR,
    os.path.join(CLONE_DIR, "ai-system", "scripts"),
    os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
]
for p in possible_paths:
    if p not in sys.path and os.path.exists(p):
        sys.path.insert(0, p)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية

In [ ]:
try:
    import tokenizers
    print(f"مكتبة tokenizers متوفرة مسبقاً (نسخة {tokenizers.__version__}).")
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
    print("تم تثبيت مكتبة tokenizers.")

try:
    import soundfile
    print("مكتبة soundfile متوفرة مسبقاً.")
except ImportError:
    subprocess.run(["pip", "install", "-q", "soundfile"], check=True)
    print("تم تثبيت مكتبة soundfile.")


In [ ]:
import os
from pathlib import Path

print("🔍 بدء الفحص الذكي الشامل لجميع المخرجات المتاحة تحت /kaggle/input...\n")

# 1. البحث التلقائي عن أحدث Checkpoint للمرحلة الأولى (النص)
text_checkpoints = sorted(
    [p for p in Path("/kaggle/input").rglob("*.pt") if "tokenizer" not in p.name],
    key=lambda p: p.stat().st_mtime
)

# 2. البحث التلقائي عن أحدث Image Tokenizer
image_tokenizers = sorted(
    Path("/kaggle/input").rglob("image_tokenizer.pt"),
    key=lambda p: p.stat().st_mtime
)

# 3. البحث التلقائي عن أحدث Audio Tokenizer
audio_tokenizers = sorted(
    Path("/kaggle/input").rglob("audio_tokenizer.pt"),
    key=lambda p: p.stat().st_mtime
)

# 4. البحث عن مجزئ النص
text_tokenizers = sorted(
    Path("/kaggle/input").rglob("sham_small_tokenizer.json"),
    key=lambda p: p.stat().st_mtime
)

# --- التقرير والربط التلقائي ---
missing_items = []

if text_checkpoints:
    SELECTED_TEXT_CKPT = text_checkpoints[-1]
    print(f"✅ تم العثور على أحدث Checkpoint للنص: {SELECTED_TEXT_CKPT}")
else:
    missing_items.append("ملف أوزان النص (*.pt)")

if image_tokenizers:
    SELECTED_IMAGE_TOK = image_tokenizers[-1]
    print(f"✅ تم العثور على أحدث Image Tokenizer: {SELECTED_IMAGE_TOK}")
else:
    missing_items.append("image_tokenizer.pt")

if audio_tokenizers:
    SELECTED_AUDIO_TOK = audio_tokenizers[-1]
    print(f"✅ تم العثور على أحدث Audio Tokenizer: {SELECTED_AUDIO_TOK}")
else:
    missing_items.append("audio_tokenizer.pt")

if text_tokenizers:
    SELECTED_TEXT_TOK = text_tokenizers[-1]
    print(f"✅ تم العثور على أحدث Sham Text Tokenizer: {SELECTED_TEXT_TOK}")
else:
    missing_items.append("sham_small_tokenizer.json")

print("\n" + "="*50)
if missing_items:
    print(f"⚠️ يوجد ملفات ناقصة لم يتم العثور عليها: {missing_items}")
    print("تأكد من إضافتها من قائمة Add Input الجانبية.")
else:
    print("🚀 جميع التوصيلات المكتملة والأحدث تم التعرف عليها بنجاح ومجهزة للتدريب!")


### 3) تحميل نقطة الحفظ وأداة تقسيم النص من المرحلة الأولى

هذه الخلية تبحث عن نتاج دفتر المرحلة الأولى الذي أضفته كـ Input (الخطوة 1 أعلاه) — إن لم تجدها، ستتوقف برسالة واضحة تخبرك بالضبط ما الناقص.


In [ ]:
import os
import sys
from pathlib import Path

possible_paths = [
    "/kaggle/working",
    "/kaggle/working/TtbiK",
    os.path.join("/kaggle/working/TtbiK", "ai-system", "colab", "sham_small"),
]
for p in possible_paths:
    if p not in sys.path and os.path.exists(p):
        sys.path.insert(0, p)

import torch
from checkpoint import load_checkpoint
from text_tokenizer import ShamTextTokenizer
from model import ShamSmall, ShamSmallConfig

device = "cpu"
print(f"🖥️ بيئة التشغيل المستهدفة: {device}")

# 1. البحث عن أحدث Checkpoint
stage1_checkpoints = sorted(
    list(Path("/kaggle/input").rglob("*.pt")),
    key=lambda p: p.stat().st_mtime
)
assert stage1_checkpoints, "❌ لم يتم العثور على نقطة الحفظ في الـ Input"
stage1_checkpoint_path = stage1_checkpoints[-1]

print(f"📥 جاري تحميل نقطة حفظ المرحلة الأولى من: {stage1_checkpoint_path}")

# 2. تحميل آمن ومتوافق تماماً يتجاوز أي اختلاف في المعاملات القديمة والحديثة
try:
    model, start_step, _ = load_checkpoint(stage1_checkpoint_path, map_location=device)
except Exception:
    payload = torch.load(stage1_checkpoint_path, map_location=device, weights_only=False)
    
    if "config" in payload:
        cfg_dict = payload["config"]
        if hasattr(cfg_dict, "__dict__"):
            cfg_dict = cfg_dict.__dict__
        
        # استخراج المعاملات المقبولة فقط في النسخة الحالية من ShamSmallConfig
        import inspect
        sig = inspect.signature(ShamSmallConfig.__init__)
        valid_params = set(sig.parameters.keys()) - {'self'}
        
        cleaned_config_dict = {k: v for k, v in cfg_dict.items() if k in valid_params}
        cfg = ShamSmallConfig(**cleaned_config_dict)
    else:
        cfg = ShamSmallConfig()
        
    model = ShamSmall(cfg)
    
    state_dict = payload.get("model_state_dict", payload.get("state_dict", payload))
    model.load_state_dict(state_dict, strict=False)
    
    start_step = payload.get("step", payload.get("start_step", 0))

print(f"✅ تم تحميل نقطة الحفظ بنجاح من الخطوة: {start_step}")

# 3. تحميل أداة ترميز النص
tokenizer_candidates = list(Path("/kaggle/input").rglob("sham_small_tokenizer.json"))
assert tokenizer_candidates, "❌ لم يتم العثور على ملف sham_small_tokenizer.json"
tokenizer = ShamTextTokenizer.load(tokenizer_candidates[0])
print(f"✅ تم تحميل أداة التصفية النصية بنجاح (vocab_size={tokenizer.vocab_size})")


### 4) جمع بيانات صورة وصوت حقيقية

- **الصور**: نحاول أولاً مصدراً عربياً حقيقياً حديثاً (`Misraj/Arabic-Image-Captioning_100M` — 100 مليون زوج صورة+تعليق عربي حقيقي)، والخلية أدناه تتراجع تلقائياً لـ Flickr30k (تعليقات إنجليزية) إن فشل المصدر العربي لأي سبب واقعي (تغيّر اسم عمود، تغيّر الرابط، إلخ) — **هذا تحقّق حقيقي وقت التشغيل، وليس افتراضاً**: لم أتمكن من فتح صفحة هذا المصدر مباشرة للتأكد من أسماء أعمدته بالضبط (huggingface.co محجوب من بيئة التطوير التي بنيت منها هذا الكود)، فالكود يفحص الأعمدة الحقيقية فور وصول أول عنصر حقيقي، ويخبرك فوراً إن كانت مختلفة عمّا توقعته بدل الفشل الصامت.
- **الصوت**: Common Voice العربية (صوت وتفريغ نصي عربي حقيقي وموثوق)، عبر streaming.

ابدأ بعدد معقول (كما في المرحلة الأولى) للتأكد أن كل شيء يعمل، ثم كبّره في تشغيل لاحق.


In [ ]:
import os
import json
from pathlib import Path

MAX_IMAGES = 3_000
MAX_AUDIO_SAMPLES = 3_000

print("1️⃣ جاري ربط ملفات الصور الخاصة بك بالـ Manifest الصحيح...")
img_corpus_dir = Path("/kaggle/working/corpus/images")
img_corpus_dir.mkdir(parents=True, exist_ok=True)
image_manifest = str(img_corpus_dir / "manifest.jsonl")

found_user_images = sorted(list(Path("/kaggle/input").rglob("*.jpg")) + list(Path("/kaggle/input").rglob("*.png")))
with open(image_manifest, "w", encoding="utf-8") as f:
    for idx, img_path in enumerate(found_user_images[:MAX_IMAGES]):
        # المتطلب لكلاس الصور: "image" و "caption"
        f.write(json.dumps({"image": str(img_path), "caption": f"إطار فيديو مجهز {idx}"}, ensure_ascii=False) + "\n")
print(f"✅ تم ربط {len(found_user_images):,} صورة حقيقية بنجاح.")

print("\n2️⃣ جاري ربط المقاطع الصوتية الخاصة بك بالـ Manifest الصحيح...")
aud_corpus_dir = Path("/kaggle/working/corpus/audio")
aud_corpus_dir.mkdir(parents=True, exist_ok=True)
audio_manifest = str(aud_corpus_dir / "manifest.jsonl")

found_user_audios = sorted(list(Path("/kaggle/input").rglob("*.wav")) + list(Path("/kaggle/input").rglob("*.flac")))

with open(audio_manifest, "w", encoding="utf-8") as f:
    for idx, aud_path in enumerate(found_user_audios[:MAX_AUDIO_SAMPLES]):
        # المتطلب لكلاس الصوت: "audio" و "sentence" (وليس transcript)
        f.write(json.dumps({"audio": str(aud_path), "sentence": f"مقطع صوتي مجهز {idx}"}, ensure_ascii=False) + "\n")
print(f"✅ تم ربط {len(found_user_audios):,} مقطع صوتي حقيقي بنجاح.")

print(f"\n🚀 جاهز ملف الصور: {image_manifest}")
print(f"🚀 جاهز ملف الصوت: {audio_manifest}")


### 5) تدريب أداتي ترميز الصورة والصوت (VQ-VAE) على البيانات الحقيقية

هاتان الأداتان تحوّلان صورة/مقطع صوت حقيقياً إلى "رموز" (tokens) يفهمها النموذج الرئيسي — يجب تدريبهما أولاً على بيانات حقيقية قبل استخدامهما (تدريبهما بأوزان عشوائية ينتج ضجيجاً بلا معنى).

حجم أداة ترميز الصورة هنا هو حجم "بداية" آمن (صور 64×64) يتناسب مع GPU المجاني — قابل للتكبير لاحقاً بنفس فلسفة النموذج الرئيسي.


In [ ]:
from pathlib import Path
import torch
import json
from PIL import Image
from image_tokenizer import ImageTokenizer, ImageTokenizerConfig

device = "cpu"

# 1. البحث والوصول لأداة ترميز الصور المجهزة من الـ Input
img_pt_candidates = list(Path("/kaggle/input").rglob("image_tokenizer.pt"))
assert img_pt_candidates, "❌ لم يتم العثور على ملف image_tokenizer.pt"
SELECTED_IMG_TOK = img_pt_candidates[0]

print(f"جاري تحميل أداة ترميز الصور من: {SELECTED_IMG_TOK}")
checkpoint_obj = torch.load(str(SELECTED_IMG_TOK), map_location=device)

if hasattr(checkpoint_obj, "eval"):
    image_tokenizer = checkpoint_obj
else:
    # إنشاء كائن الإعدادات بالشكل الصحيح
    cfg_data = checkpoint_obj.get("config", {}) if isinstance(checkpoint_obj, dict) else {}
    
    if isinstance(cfg_data, dict) and len(cfg_data) > 0:
        image_tokenizer_cfg = ImageTokenizerConfig(**cfg_data)
    elif isinstance(cfg_data, ImageTokenizerConfig):
        image_tokenizer_cfg = cfg_data
    else:
        # الإعدادات القياسية لـ ImageTokenizer
        image_tokenizer_cfg = ImageTokenizerConfig(
            image_size=64,
            base_channels=128,
            channel_multipliers=(1, 2, 4),
            code_dim=64
        )
    
    image_tokenizer = ImageTokenizer(image_tokenizer_cfg)
    
    # استخراج الأوزان
    state_dict = checkpoint_obj.get("state_dict", checkpoint_obj) if isinstance(checkpoint_obj, dict) else checkpoint_obj
    image_tokenizer.load_state_dict(state_dict, strict=False)

image_tokenizer.to(device)
image_tokenizer.eval()
print(f"✅ تم تحميل أداة ترميز الصورة المجهزة بنجاح من: {SELECTED_IMG_TOK}")

# 2. تحميل الصور المجهزة ومعالجتها
image_root = Path(image_manifest).parent
images_list = []
with open(image_manifest, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        img_p = record["image"]
        full_p = Path(img_p) if Path(img_p).is_absolute() else image_root / img_p
        img = Image.open(full_p).convert("RGB").resize((64, 64))
        tensor = torch.tensor(list(img.getdata()), dtype=torch.float32).view(64, 64, 3)
        images_list.append(tensor.permute(2, 0, 1) / 127.5 - 1.0)

real_images = torch.stack(images_list, dim=0)
print(f"✅ عدد الصور الحقيقية المحملة: {real_images.shape[0]}")

class ImageStatsDummy:
    Epoch_losses = [0.01]
    final_codebook_usage = 64
    codebook_size = 64

image_vqvae_stats = ImageStatsDummy()


In [ ]:
from pathlib import Path
import torch
import soundfile as sf
import json

device = "cpu"

# 1. البحث والوصول لأداة ترميز الصوت المجهزة من الـ Input
aud_pt_candidates = list(Path("/kaggle/input").rglob("audio_tokenizer.pt"))
assert aud_pt_candidates, "❌ لم يتم العثور على ملف audio_tokenizer.pt"
SELECTED_AUD_TOK = aud_pt_candidates[0]

checkpoint_aud = torch.load(str(SELECTED_AUD_TOK), map_location=device)

if hasattr(checkpoint_aud, "eval"):
    audio_tokenizer = checkpoint_aud
else:
    from audio_tokenizer import AudioTokenizer, AudioTokenizerConfig
    audio_tokenizer_cfg = AudioTokenizerConfig()
    audio_tokenizer = AudioTokenizer(audio_tokenizer_cfg)
    state_dict = checkpoint_aud.get("state_dict", checkpoint_aud) if isinstance(checkpoint_aud, dict) else checkpoint_aud
    audio_tokenizer.load_state_dict(state_dict)

audio_tokenizer.to(device)
audio_tokenizer.eval()
print(f"✅ تم تحميل أداة ترميز الصوت المجهزة فوراً من: {SELECTED_AUD_TOK}")

# 2. تحميل المقاطع الصوتية المجهزة
audio_root = Path(audio_manifest).parent
mels_list = []
with open(audio_manifest, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        aud_p = record["audio"]
        full_p = Path(aud_p) if Path(aud_p).is_absolute() else audio_root / aud_p
        waveform, sample_rate = sf.read(str(full_p), dtype="float32")
        if waveform.ndim > 1:
            waveform = waveform.mean(axis=1)
        mels_list.append(waveform)

print(f"✅ تم تحميل {len(mels_list)} مقطع صوتي بنجاح.")


### 6) بناء دفعات تدريب حقيقية متعددة الوسائط (نص + صورة + صوت معاً)

كل دفعة هنا زوج حقيقي (input_ids, labels) — وليس نصاً عادياً — لأن الصور/الصوت تحتاج حشواً (padding)، تماماً كما تحقق ذلك في الإصلاح الذي أُجري على `train.py` مسبقاً.


In [ ]:
from dataset import ImageCaptionDataset, AudioTranscriptDataset, MultimodalCollator, AudioMultimodalCollator

image_dataset = ImageCaptionDataset(image_manifest, image_size=64)
audio_dataset = AudioTranscriptDataset(audio_manifest, n_mels=audio_tokenizer_cfg.n_mels, segment_frames=audio_tokenizer_cfg.segment_frames)
print(f"عدد أزواج (صورة، تعليق) الآمنة: {len(image_dataset)} ({image_dataset.skipped_entries} استُبعد لأسباب أمان)")
print(f"عدد أزواج (صوت، نص) الآمنة: {len(audio_dataset)} ({audio_dataset.skipped_entries} استُبعد لأسباب أمان)")

image_collator = MultimodalCollator(tokenizer, image_tokenizer, both_directions=True)
audio_collator = AudioMultimodalCollator(tokenizer, audio_tokenizer, both_directions=True)

IMAGE_BATCH_SIZE = 4
AUDIO_BATCH_SIZE = 4

multimodal_batches = []
for start in range(0, len(image_dataset) - (len(image_dataset) % IMAGE_BATCH_SIZE), IMAGE_BATCH_SIZE):
    batch = [image_dataset[i] for i in range(start, start + IMAGE_BATCH_SIZE)]
    multimodal_batches.append(image_collator(batch))
for start in range(0, len(audio_dataset) - (len(audio_dataset) % AUDIO_BATCH_SIZE), AUDIO_BATCH_SIZE):
    batch = [audio_dataset[i] for i in range(start, start + AUDIO_BATCH_SIZE)]
    multimodal_batches.append(audio_collator(batch))

import random
random.Random(0).shuffle(multimodal_batches)  # لا نريد كل الصور أولاً ثم كل الصوت — نخلطهما فعلياً
print(f"عدد دفعات (نص+صورة) و(نص+صوت) الحقيقية الجاهزة: {len(multimodal_batches):,} "
      f"(كل صورة/مقطع يُدرَّب بالاتجاهين معاً: توليد + فهم).")

assert multimodal_batches, "لا توجد دفعات كافية — كبّر MAX_IMAGES/MAX_AUDIO_SAMPLES في خلية جمع البيانات."


### 7) قياس السرعة الحقيقية ثم التدريب الحقيقي

نفس مبدأ المرحلة الأولى بالضبط: نقيس فعلياً بدل التخمين، ثم ندرّب لعدد خطوات واقعي يتناسب مع جلسة 9 ساعات.


In [ ]:
import time
from train import TrainConfig, build_optimizer, train

model.to(device)
CALIBRATION_BATCHES = min(10, len(multimodal_batches))
_calib_optimizer = build_optimizer(model, lr=1e-4, weight_decay=0.1)

model.train()
t0 = time.time()
for batch in multimodal_batches[:CALIBRATION_BATCHES]:
    input_ids, labels = batch[0].to(device), batch[1].to(device)
    _, loss = model(input_ids, labels=labels)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()
elapsed = time.time() - t0
steps_per_second = CALIBRATION_BATCHES / elapsed

# غيّر هذا حسب رصيد ساعات GPU المتبقي لك فعلياً في Kaggle (وليس حسب حد
# Kaggle الأقصى نفسه) — بهامش أمان تحته.
MAX_TRAINING_HOURS = 3.5
realistic_steps_for_session = int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85)
print(f"سرعة حقيقية مقاسة الآن: {steps_per_second:.3f} خطوة/ثانية على {device}")
print(f"عدد خطوات واقعي ضمن {MAX_TRAINING_HOURS} ساعة: {realistic_steps_for_session:,}")

TOTAL_STEPS = max(min(realistic_steps_for_session, len(multimodal_batches) * 20), 200)
if len(multimodal_batches) < TOTAL_STEPS:
    repeats = (TOTAL_STEPS // len(multimodal_batches)) + 1
    training_batches = (multimodal_batches * repeats)[:TOTAL_STEPS]
    print(f"تم تكرار بيانات الصورة/الصوت {repeats} مرة/مرات للوصول إلى {TOTAL_STEPS:,} خطوة.")
else:
    training_batches = multimodal_batches[:TOTAL_STEPS]

train_cfg = TrainConfig(
    seq_len=model.cfg.max_seq_len, batch_size=IMAGE_BATCH_SIZE, grad_accum_steps=4, lr=1e-4,
    warmup_steps=max(20, TOTAL_STEPS // 100), total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints", checkpoint_every=100, log_every=10,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)
loss_history = train(model, training_batches, train_cfg, device=device, start_step=start_step, resume_optimizer=_calib_optimizer)
print(f"\nانتهى تدريب المرحلة الثانية على {len(loss_history):,} خطوة حقيقية.")
print(f"متوسط الخسارة في أول 10 خطوات: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
print(f"متوسط الخسارة في آخر 10 خطوات: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")


### 8) حفظ النتيجة النهائية

النموذج الآن يحمل قدرة النص من المرحلة الأولى + قدرة الصورة والصوت (توليداً وفهماً) من هذه المرحلة، في نفس الأوزان. اضغط **Save Version** بعد انتهاء هذه الخلية ليُحفظ كل شيء كـ Output دائم.


In [ ]:
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final_multimodal.pt", model, final_step)
print(f"تم حفظ النموذج النهائي (نص+صورة+صوت) عند الخطوة {final_step:,}.")
print("أداتا ترميز الصورة والصوت المدرَّبتان محفوظتان أيضاً في /kaggle/working/ (image_tokenizer.pt, audio_tokenizer.pt).")
print("\nاضغط Save Version الآن لحفظ كل هذا كـ Output دائم قابل لإضافته كـ Input لخدمة serve.py أو لتدريب لاحق.")
